# Top-k log-prob drop: ContextCite vs cosine baseline

Reads the JSON produced by `cosine_baseline.py` and plots the response log-prob
drop after removing each method's top-k sources. A larger drop = the method
picked sources the answer truly depends on.

Run the data step first (the slow LLM part):

```bash
python demo/eval/cosine_baseline.py            # writes results/test_qa_en_baseline.json
```

Needs matplotlib: `pip install matplotlib`.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Point this at the JSON cosine_baseline.py wrote.
RESULTS = Path("results/test_qa_en_baseline.json")
data = json.loads(RESULTS.read_text())
ev = data["eval"]
ks = ev["k_values"]
methods = list(ev["methods"])  # e.g. cc_32, cc_64, cc_128, similarity
print("mock:", data["meta"]["mock"], "| baseline log-prob:", round(ev["baseline_logprob"], 2))
print("methods:", methods, "| ks:", ks)

In [ ]:
# Grouped, color-coded bar chart: x = k, one colored bar per method.

def pretty(name):
    return f"ContextCite @{name[3:]}" if name.startswith("cc_") else "Cosine similarity"

# Stable colors: a shaded family for the ContextCite budgets, a contrasting
# color for the baseline.
cc = [m for m in methods if m.startswith("cc_")]
blues = plt.cm.Blues(np.linspace(0.5, 0.9, len(cc)))
colors = {m: blues[i] for i, m in enumerate(cc)}
for m in methods:
    if m not in colors:
        colors[m] = "#e8703a"  # similarity / any non-cc method

x = np.arange(len(ks))
width = 0.8 / len(methods)

fig, ax = plt.subplots(figsize=(8, 5))
for j, m in enumerate(methods):
    drops = [ev["methods"][m]["drops"][str(k)] for k in ks]
    offset = (j - (len(methods) - 1) / 2) * width
    ax.bar(x + offset, drops, width, label=pretty(m), color=colors[m])

ax.set_xticks(x, [f"k={k}" for k in ks])
ax.set_xlabel("Top-k sources removed")
ax.set_ylabel("Response log-prob drop")
ax.set_title(f"Top-k log-prob drop — {data['meta']['mock']}")
ax.legend(title="Method")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig("results/logprob_drop.png", dpi=150)
plt.show()